# Nile-Chat-12B (Base) — Query Router / Intent-Classification Sidecar Server

Serves the RAW, unadapted `MBZUAI-Paris/Nile-Chat-12B` checkpoint as an
OpenAI-compatible `/v1/chat/completions` endpoint via vLLM, dedicated **only**
to `IntentRoutingController`'s two query-router calls — `classify_intent()` and
`rewrite_query()` (see `src/stores/query_router/providers/NileChat12BBaseProvider.py`)
— never used for Mode A reply generation, which stays on the separately-deployed
**fine-tuned** 12B checkpoint (`stores/generation/providers/NileChatProvider.py`).

**Base, not fine-tuned — deliberately.** This is the same base weights Mode A's
own generation client was LoRA-fine-tuned FROM, served here in its raw,
general-purpose form. 2026-09-01: replaces the earlier `NileChat4BProvider`
entirely (deleted). Real production evidence — four separate prompt-engineering
attempts, all failing the same generic-reference anaphora-resolution pattern —
showed that 4B checkpoint hitting a genuine reliability ceiling on this task; see
`README.md`'s "Dual-model architecture" section for the full history. This
notebook's serving cells are adapted from `src/run_nilechat12b.ipynb` (Phase 0's
own bake-off notebook for this exact base checkpoint) — same proven install
fixes, same serving pattern, different port/served-model-name so it doesn't
collide with a simultaneously-running Mode A deployment.

**Before running anything**: `Runtime > Change runtime type > A100 GPU`, then
confirm the GPU is actually attached with the cell below.

## 1. Confirm the GPU Colab actually gave you

In [ ]:
!nvidia-smi


If the cell above errors or shows no GPU, stop here and fix the runtime type before continuing — every cell after this one assumes a real GPU is visible.

In [ ]:
!pip install -q -U vllm transformers accelerate


**Fix for the `torchaudio`/`torchvision`/`torch` CUDA-version saga** — proven
in `run_nilechat12b.ipynb`'s own Phase 0 bake-off run against this exact base
checkpoint, reused verbatim here:

1. `vllm.transformers_utils.processors.*` imports `torchvision`
   **unconditionally** at package-import time, for every architecture vLLM
   supports, not just the one being served — a real, unguarded hard
   requirement, confirmed by a live `ModuleNotFoundError` at server startup
   before any weights loaded.
2. `torchaudio` is different: every real failure seen has been a
   *present-but-CUDA-mismatched* `RuntimeError`, never a clean
   `ModuleNotFoundError` from being absent — so the fix is to reinstall
   `torchvision` matched to whatever CUDA tag `torch` actually resolved to
   (detected live, never hardcoded), and leave `torchaudio` **uninstalled**
   entirely. That converts an uncatchable crash into a cleanly-catchable
   optional-dependency miss.

**If resuming a Colab session where an earlier cell already crashed vLLM**:
`Runtime > Restart session` first, then run from the top.

In [ ]:
# Detect whatever torch build vLLM actually installed, reinstall torchvision (a confirmed
# hard requirement) matched to that exact CUDA-tagged index, and deliberately leave torchaudio
# UNINSTALLED — see the note above for why that's the correct fix, not a workaround.
import torch

torch_version = torch.__version__.split("+")[0]   # strip local suffix, e.g. "2.9.0+cu130" -> "2.9.0"
torch_cuda = torch.version.cuda                    # whatever vLLM actually pulled in
cuda_tag = "cu" + torch_cuda.replace(".", "")       # "13.0" -> "cu130"

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

# Verify in a fresh subprocess (not this kernel) so a stale sys.modules cache can't hide a
# real remaining problem or falsely report one that's already fixed. Only torch/torchvision are
# required to import cleanly — torchaudio being ABSENT is the intended, correct end state.
import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print('torch:', torch.__version__, torch.version.cuda); "
     "print('torchvision:', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.")


**Plan B — only run the cell below if `vllm serve` (further down) crashes with `ModuleNotFoundError: No module named 'torchaudio'`.** That would mean torchaudio is *also* a real, unguarded hard requirement, the same way `torchvision` turned out to be. Don't run this pre-emptively — only if that specific error actually appears.

In [ ]:
# PLAN B — see the markdown note above. Writes a minimal real stub package to site-packages
# so any bare `import torchaudio` succeeds without needing a real (CUDA-build-sensitive)
# install. Safe here specifically because this whole pipeline is text-only classification/
# rewriting — nothing should ever call a real torchaudio function.
import sysconfig, os

site_packages = sysconfig.get_paths()["purelib"]
stub_dir = os.path.join(site_packages, "torchaudio")
os.makedirs(stub_dir, exist_ok=True)
with open(os.path.join(stub_dir, "__init__.py"), "w") as f:
    f.write(
        "__version__ = '0.0.0-stub'\n"
        "# Minimal stub -- no real matched-CUDA torchaudio build exists yet, and this\n"
        "# text-only pipeline never calls a real torchaudio function. Exists only to\n"
        "# satisfy an unguarded `import torchaudio` statement somewhere in vLLM's startup path.\n"
    )
print(f"Wrote stub torchaudio package to {stub_dir} -- re-run the vllm serve cell now.")


## 2. Serve the base checkpoint via vLLM

- **`--dtype bfloat16`** — the dtype Nile-Chat-12B (Gemma3-based) is actually
  published/trained in; requires compute capability ≥ 8.0, which an A100 has.
- **`--max-model-len 8192`** — this base checkpoint's real published context
  length, same as Mode A's own fine-tuned sibling (same underlying model).
  classify_intent/rewrite_query's own prompts are small regardless (a few
  hundred tokens even with a full 6-message history) — this ceiling is not
  expected to bind, unlike the retired 4B provider's tight 2048-token budget,
  which was the direct cause of a real production 400 earlier in this project.
- **No `--enforce-eager`** — that flag was specifically needed for Falcon-H1's
  hybrid Mamba2 architecture (CUDA graph capture SIGKILLed the process — see
  README's Phase 0 verdict). Nile-Chat-12B is a standard dense Gemma3
  transformer with no comparable known gap.
- **`--port 8002`, `--served-model-name nile-chat-12b-base`** — deliberately
  different from `run_nilechat12b.ipynb`'s own `8001`/`nile-chat-12b`, so this
  can run alongside a Mode A generation deployment on the same machine without
  colliding. `nile-chat-12b-base` must match `QUERY_ROUTER_MODEL_NAME` in
  `src/.env` exactly, or every request from `NileChat12BBaseProvider` 404s.

In [ ]:
!nohup vllm serve "MBZUAI-Paris/Nile-Chat-12B" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8002 \
    --served-model-name nile-chat-12b-base \
    > vllm.log 2>&1 &


Poll until the server is up, then a raw smoke test.

In [ ]:
# Poll instead of a fixed sleep -- a 12B model can take a while to load.
import time

ready = False
for attempt in range(60):  # up to 10 minutes
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log or "ValidationError" in log:
        print("vLLM logged an error while loading -- check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 60 vllm.log
print("\n--- server ready:", ready, "---\n")

!curl -s http://localhost:8002/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"nile-chat-12b-base","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'


If the curl call above didn't return a real completion, stop and fix it before opening a tunnel — a tunnel just exposes whatever's on :8002, broken or not.

## 3. Expose via Cloudflare Tunnel

Using **cloudflared**, not ngrok/localtunnel — matches this project's own
already-proven pattern for every other Colab-served model here, and needs no
signup/authtoken.

In [ ]:
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed -- re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8002 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s -- check cloudflared.log for errors and re-run this cell"

print("=" * 70)
print("Tunnel is live. Paste these EXACT lines into your local src/.env:")
print("=" * 70)
print(f"QUERY_ROUTER_BASE_URL={tunnel_url}")
print("QUERY_ROUTER_MODEL_NAME=nile-chat-12b-base")
print("QUERY_ROUTER_BACKEND=NILE_CHAT_12B_BASE")
print("QUERY_ROUTER_REQUEST_TIMEOUT_SECONDS=20")
print("=" * 70)
print("Then restart your local app process so main.py\'s startup_span picks up")
print("the new QUERY_ROUTER_BASE_URL and builds app.query_router_client.")
print("=" * 70)


## 4. Verify with the exact `classify_intent` / `rewrite_query` prompt shapes

Sends the same real scenario this project's own multi-turn testing has used
throughout (short reply "اه" after the assistant asked a specific branch
question), through the same TWO separate message structures
`NileChat12BBaseProvider.classify_intent` and
`NileChat12BBaseProvider.rewrite_query` actually build — using the same
centralized Bucket C directives (`whatsapp_intent_classification_directive`,
`whatsapp_cqr_directive`, `system_directives.py`) — so you can see with your own
eyes whether this base checkpoint reliably resolves the short reply, before
wiring it into the real app.

In [ ]:
import json
import re
import requests

_JSON_OBJECT_RE = re.compile(r"\{[^{}]*\}")

allowed_intents = ["complaint", "inquiry", "book_appointment"]

history_lines = "Assistant: تحب تعرف مواعيد فرع المهندسين؟"
final_message = "اه"

# --- Call A: classify_intent (must match NileChat12BBaseProvider.classify_intent's
# own prompt shape, using whatsapp_intent_classification_directive's real content) ---
classify_system_prompt = "\n".join([
    "You are a senior intent classifier for an Egyptian Arabic medical-services WhatsApp "
    "assistant.",
    "First, in a <reasoning>...</reasoning> block, briefly reason in 1-2 sentences about what "
    "the patient is actually asking for semantically, and whether the final message on its own "
    "already signals a clear intent or depends on the conversation above it.",
    "After the </reasoning> block, on a new line, output a single JSON object of the exact "
    "shape: {\"intent\": \"<one value>\"} and nothing else after it. Never wrap it in a code "
    "fence, never output it more than once, never add any text after it.",
    "Classify the final message as given — it is already standalone, so classify it directly "
    "without needing to resolve anything against the conversation history.",
    f"<one value> MUST be exactly one of: {json.dumps(allowed_intents, ensure_ascii=False)}",
])
classify_user_content = "\n".join([
    "## Recent conversation (context only):", history_lines, "",
    "## Message to classify (already standalone):", final_message,
])

response_a = requests.post(
    "http://localhost:8002/v1/chat/completions",
    json={
        "model": "nile-chat-12b-base",
        "messages": [
            {"role": "system", "content": classify_system_prompt},
            {"role": "user", "content": classify_user_content},
        ],
        "temperature": 0.0,
        "max_tokens": 150,
        "stop": ["<end_of_turn>"],
    },
    timeout=30,
)
response_a.raise_for_status()
raw_a = response_a.json()["choices"][0]["message"]["content"].strip()

print("=== CALL A: classify_intent ===")
print("--- RAW MODEL OUTPUT ---")
print(raw_a)
print("------------------------")

matches_a = _JSON_OBJECT_RE.findall(raw_a)
intent = None
if matches_a:
    try:
        intent = json.loads(matches_a[-1]).get("intent")
    except (json.JSONDecodeError, AttributeError):
        pass
print(f"Parsed intent: {intent!r}")
intent_ok = intent in allowed_intents
print("PASS: valid intent." if intent_ok else "FAIL: missing/invalid intent.")
print()

# --- Call B: rewrite_query (must match NileChat12BBaseProvider.rewrite_query's
# own prompt shape, using whatsapp_cqr_directive's real content) ---
rewrite_system_prompt = "\n".join([
    "You are a senior query-rewriting assistant for an Egyptian Arabic medical-services "
    "WhatsApp assistant, preparing a patient\'s message for a knowledge-base search.",
    "First, in a <reasoning>...</reasoning> block, briefly reason in 1-2 sentences about "
    "whether the final patient message is already a complete, standalone question or "
    "statement on its own, or depends on the conversation above it to make sense.",
    "After the </reasoning> block, on a new line, output a single JSON object of the exact "
    "shape: {\"resolved_query\": \"<string>\"} and nothing else after it. Never wrap it in a "
    "code fence, never output it more than once, never add any text after it.",
    "resolved_query: rewrite the final patient message into a short, self-contained, standalone "
    "version that makes sense with NO prior conversation attached — it is used to search a "
    "knowledge base, so it must capture what the patient actually wants to know or confirm, "
    "using the recent conversation only to fill in what the message alone doesn\'t say. Never "
    "invent facts, options, or details the conversation doesn\'t already contain. If the final "
    "patient message is already a complete, standalone question or statement, resolved_query is "
    "that same message unchanged, WORD FOR WORD — never add a question word, a time frame, or "
    "any other framing the patient didn\'t use, even if it feels like a natural rephrasing.",
])
rewrite_user_content = "\n".join([
    "## Recent conversation (context only):", history_lines, "",
    "## Final patient message to rewrite:", final_message,
])

response_b = requests.post(
    "http://localhost:8002/v1/chat/completions",
    json={
        "model": "nile-chat-12b-base",
        "messages": [
            {"role": "system", "content": rewrite_system_prompt},
            {"role": "user", "content": rewrite_user_content},
        ],
        "temperature": 0.0,
        "max_tokens": 180,
        "stop": ["<end_of_turn>"],
    },
    timeout=30,
)
response_b.raise_for_status()
raw_b = response_b.json()["choices"][0]["message"]["content"].strip()

print("=== CALL B: rewrite_query ===")
print("--- RAW MODEL OUTPUT ---")
print(raw_b)
print("------------------------")

matches_b = _JSON_OBJECT_RE.findall(raw_b)
resolved_query = None
if matches_b:
    try:
        resolved_query = json.loads(matches_b[-1]).get("resolved_query")
    except (json.JSONDecodeError, AttributeError):
        pass
print(f"Parsed resolved_query: {resolved_query!r}")
rewrite_ok = (
    isinstance(resolved_query, str) and resolved_query.strip() and resolved_query.strip() != final_message
)
print("\nPASS: both calls succeeded, short reply correctly resolved." if (intent_ok and rewrite_ok) else
      "\nCHECK CAREFULLY: at least one call didn\'t parse or resolve correctly (see values above).")


## 5. Keep this running

- The tunnel URL is **ephemeral** — changes every time this notebook's runtime
  restarts or disconnects. Re-run from the serving cell onward for a new URL,
  update `QUERY_ROUTER_BASE_URL` in `src/.env` again, restart your local app.
- Development/testing setup, not a production deployment.
- An A100 Colab runtime is billed while alive — shut it down when done testing.
- Diagnostics if anything above looked wrong:

In [ ]:
!ps aux | grep vllm
print("---")
!nvidia-smi
